In [1]:
import qdrant_client
from langchain_qdrant import QdrantVectorStore
from langchain_ollama import OllamaEmbeddings, ChatOllama
import networkx as nx

# 1. Verifica connessione a Qdrant
client = qdrant_client.QdrantClient(url="http://localhost:6333")
print("✅ Qdrant connesso! Collezioni:", client.get_collections())

# 2. Verifica Nomic Embeddings
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector = embeddings.embed_query("Test vettorizzazione Nomic")
print(f"✅ Nomic attivo! Dimensione vettore generato: {len(vector)}")

# 3. Verifica Llama 3.1
llm = ChatOllama(model="llama3.1")
response = llm.invoke("Rispondi solo con la parola 'PRONTO'")
print("✅ Llama 3.1 attivo! Risposta:", response.content.strip())

# 4. Verifica NetworkX
G = nx.Graph()
G.add_edge("Chunk_A", "Chunk_B", relation="test")
print("✅ NetworkX attivo! Nodi di prova:", list(G.nodes))

✅ Qdrant connesso! Collezioni: collections=[]
✅ Nomic attivo! Dimensione vettore generato: 768
✅ Llama 3.1 attivo! Risposta: PRONTO
✅ NetworkX attivo! Nodi di prova: ['Chunk_A', 'Chunk_B']


In [2]:
from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3.1", temperature=0)
print(llm.invoke("Rispondi con una sola parola: funzioni?"))

content='Ufficio' additional_kwargs={} response_metadata={'model': 'llama3.1', 'created_at': '2026-09-03T09:08:46.872899844Z', 'done': True, 'done_reason': 'stop', 'total_duration': 205621461, 'load_duration': 1574232, 'prompt_eval_count': 23, 'prompt_eval_duration': 84618000, 'eval_count': 4, 'eval_duration': 111483000, 'logprobs': None, 'model_name': 'llama3.1', 'model_provider': 'ollama'} id='lc_run--01a06687-1fc9-7532-a888-e0269f9921c3-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 23, 'output_tokens': 4, 'total_tokens': 27}


In [3]:
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

embeddings = OllamaEmbeddings(model="nomic-embed-text")
client = QdrantClient(url="http://localhost:6333")

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="tesi_graphrag_chunks",
    embedding=embeddings,
)

UnexpectedResponse: Unexpected Response: 404 (Not Found)
Raw response content:
b'{"status":{"error":"Not found: Collection `tesi_graphrag_chunks` doesn\'t exist!"},"time":0.00001275}'

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance

embeddings = OllamaEmbeddings(model="nomic-embed-text")
client = QdrantClient(url="http://localhost:6333")

test_vector = embeddings.embed_query("prova")
print(f"Dimensione embedding: {len(test_vector)}")  # dovrebbe stampare 768

if not client.collection_exists("tesi_graphrag_chunks"):
    client.create_collection(
        collection_name="tesi_graphrag_chunks",
        vectors_config=VectorParams(size=len(test_vector), distance=Distance.COSINE),
    )

vectorstore = QdrantVectorStore(
    client=client,
    collection_name="tesi_graphrag_chunks",
    embedding=embeddings,
)

Dimensione embedding: 768


In [5]:
import networkx as nx
import json
from types import SimpleNamespace

# 1. Simulazione dei chunk recuperati e dei punteggi di similarità da Qdrant
retrieved_chunks = [
    SimpleNamespace(id="chunk_0", text="GraphRAG combina Vector DB e Grafi di Conoscenza."),
    SimpleNamespace(id="chunk_1", text="Qdrant gestisce la ricerca vettoriale ad alte prestazioni."),
    SimpleNamespace(id="chunk_2", text="NetworkX modella la struttura del grafo nel backend Python.")
]

similarity_pairs = [
    ("chunk_0", "chunk_1", 0.85),
    ("chunk_0", "chunk_2", 0.78)
]

# 2. Costruzione del Grafo NetworkX
G = nx.Graph()

for chunk in retrieved_chunks:
    G.add_node(chunk.id, text=chunk.text, tags=[])

for a, b, score in similarity_pairs:
    G.add_edge(a, b, weight=score)

# 3. Conversione nel formato JSON (nodes/links) per D3.js / anywidget
graph_json = nx.node_link_data(G)

print("✅ Grafo creato con successo!")
print(f"Nodi presenti: {len(G.nodes)}")
print(f"Archi presenti: {len(G.edges)}")
print("\nStruttura JSON generata per il frontend:")
print(json.dumps(graph_json, indent=2))

✅ Grafo creato con successo!
Nodi presenti: 3
Archi presenti: 2

Struttura JSON generata per il frontend:
{
  "directed": false,
  "multigraph": false,
  "graph": {},
  "nodes": [
    {
      "text": "GraphRAG combina Vector DB e Grafi di Conoscenza.",
      "tags": [],
      "id": "chunk_0"
    },
    {
      "text": "Qdrant gestisce la ricerca vettoriale ad alte prestazioni.",
      "tags": [],
      "id": "chunk_1"
    },
    {
      "text": "NetworkX modella la struttura del grafo nel backend Python.",
      "tags": [],
      "id": "chunk_2"
    }
  ],
  "edges": [
    {
      "weight": 0.85,
      "source": "chunk_0",
      "target": "chunk_1"
    },
    {
      "weight": 0.78,
      "source": "chunk_0",
      "target": "chunk_2"
    }
  ]
}


In [6]:
import anywidget
import traitlets

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = """
    import * as d3 from "https://esm.sh/d3@7";

    function render({ model, el }) {
      el.innerHTML = ""; // Pulisce il container
      
      const width = 600;
      const height = 400;
      
      const svg = d3.select(el).append("svg")
        .attr("width", width)
        .attr("height", height)
        .style("background", "#f9f9f9")
        .style("border", "1px solid #ddd");

      function draw() {
        const graph = model.get("graph_data");
        if (!graph || !graph.nodes || graph.nodes.length === 0) return;

        svg.selectAll("*").remove(); // Pulisce i disegni precedenti

        // Clona nodi e link per la simulazione D3
        const nodes = graph.nodes.map(d => ({ ...d }));
        const links = graph.links.map(d => ({ ...d }));

        const simulation = d3.forceSimulation(nodes)
          .force("link", d3.forceLink(links).id(d => d.id).distance(100))
          .force("charge", d3.forceManyBody().strength(-200))
          .force("center", d3.forceCenter(width / 2, height / 2));

        const link = svg.append("g")
          .selectAll("line")
          .data(links)
          .enter().append("line")
          .attr("stroke", "#999")
          .attr("stroke-width", 2);

        const node = svg.append("g")
          .selectAll("circle")
          .data(nodes)
          .enter().append("circle")
          .attr("r", 12)
          .attr("fill", "#4f46e5")
          .style("cursor", "pointer")
          .on("click", (event, d) => {
             model.set("selected_tag", { id: d.id, text: d.text });
             model.save_changes();
          });

        const label = svg.append("g")
          .selectAll("text")
          .data(nodes)
          .enter().append("text")
          .text(d => d.id)
          .attr("font-size", "12px")
          .attr("dx", 15)
          .attr("dy", 4);

        simulation.on("tick", () => {
          link
            .attr("x1", d => d.source.x)
            .attr("y1", d => d.source.y)
            .attr("x2", d => d.target.x)
            .attr("y2", d => d.target.y);

          node
            .attr("cx", d => d.x)
            .attr("cy", d => d.y);

          label
            .attr("x", d => d.x)
            .attr("y", d => d.y);
        });
      }

      model.on("change:graph_data", draw);
      draw();
    }
    export default { render };
    """
    
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)

# Istanziazione e rendering
widget = ChunkGraphWidget()
widget.graph_data = json_output
widget

NameError: name 'json_output' is not defined

In [11]:
import anywidget
import traitlets
import networkx as nx

# 1. Generazione dei dati del grafo di prova
G = nx.Graph()
G.add_node("chunk_0", text="GraphRAG combina Vector DB e Grafi di Conoscenza.", tags=[])
G.add_node("chunk_1", text="Qdrant gestisce la ricerca vettoriale ad alte prestazioni.", tags=[])
G.add_node("chunk_2", text="NetworkX modella la struttura del grafo nel backend Python.", tags=[])
G.add_edge("chunk_0", "chunk_1", weight=0.85)
G.add_edge("chunk_0", "chunk_2", weight=0.78)

json_output = nx.node_link_data(G)

# 2. Definizione del Widget D3.js + anywidget
class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = """
    import * as d3 from "https://esm.sh/d3@7";

    function render({ model, el }) {
      el.innerHTML = "";
      
      const width = 600;
      const height = 400;
      
      const svg = d3.select(el).append("svg")
        .attr("width", width)
        .attr("height", height)
        .style("background", "#f9f9f9")
        .style("border", "1px solid #ddd");

      function draw() {
        const graph = model.get("graph_data");
        if (!graph || !graph.nodes || graph.nodes.length === 0) return;

        svg.selectAll("*").remove();

        const nodes = graph.nodes.map(d => ({ ...d }));
        const links = graph.links.map(d => ({ ...d }));

        const simulation = d3.forceSimulation(nodes)
          .force("link", d3.forceLink(links).id(d => d.id).distance(100))
          .force("charge", d3.forceManyBody().strength(-200))
          .force("center", d3.forceCenter(width / 2, height / 2));

        const link = svg.append("g")
          .selectAll("line")
          .data(links)
          .enter().append("line")
          .attr("stroke", "#999")
          .attr("stroke-width", 2);

        const node = svg.append("g")
          .selectAll("circle")
          .data(nodes)
          .enter().append("circle")
          .attr("r", 12)
          .attr("fill", "#4f46e5")
          .style("cursor", "pointer")
          .on("click", (event, d) => {
             model.set("selected_tag", { id: d.id, text: d.text });
             model.save_changes();
          });

        const label = svg.append("g")
          .selectAll("text")
          .data(nodes)
          .enter().append("text")
          .text(d => d.id)
          .attr("font-size", "12px")
          .attr("dx", 15)
          .attr("dy", 4);

        simulation.on("tick", () => {
          link
            .attr("x1", d => d.source.x)
            .attr("y1", d => d.source.y)
            .attr("x2", d => d.target.x)
            .attr("y2", d => d.target.y);

          node
            .attr("cx", d => d.x)
            .attr("cy", d => d.y);

          label
            .attr("x", d => d.x)
            .attr("y", d => d.y);
        });
      }

      model.on("change:graph_data", draw);
      draw();
    }
    export default { render };
    """
    
    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)

# 3. Istanziazione e rendering
widget = ChunkGraphWidget()
widget.graph_data = json_output
widget

In [1]:
%%writefile .gitignore
# 1. Cartella dati (bloccata completamente)
data/

# 2. Ambienti virtuali
.venv/
tesi_env/
*venv/
env/

# 3. Cache Python e Jupyter
__pycache__/
*.py[cod]
.ipynb_checkpoints/
.pytest_cache/

# 4. Database vettoriali, DB locali e log
qdrant_db/
*.db
*.sqlite
logs/
*.log

# 5. Documenti, dataset e modelli pesanti
*.pdf
*.csv
*.json
*.parquet
*.zip
*.tar.gz
*.txt
*.pth
*.bin
*.safetensors
*.gguf

# 6. File sensibili e chiavi API
.env
.env.*

# 7. File di sistema ed editor
.DS_Store
.vscode/
.idea/

Overwriting .gitignore


In [1]:
%%bash
cd ~/tesi_graphrag
mkdir -p src/frontend

touch src/__init__.py

cat << 'EOF' > src/frontend/chunk_graph.js
import * as d3 from "https://esm.sh/d3@7";

function render({ model, el }) {
  el.innerHTML = "";

  const width = 600;
  const height = 400;

  const svg = d3.select(el).append("svg")
    .attr("width", width)
    .attr("height", height)
    .style("background", "#f9f9f9")
    .style("border", "1px solid #cbd5e1")
    .style("border-radius", "8px");

  function draw() {
    const graph = model.get("graph_data");
    if (!graph || !graph.nodes || graph.nodes.length === 0) return;

    svg.selectAll("*").remove();

    const nodes = graph.nodes.map(d => ({ ...d }));
    const rawLinks = graph.links || graph.edges || [];
    const links = rawLinks.map(d => ({ ...d }));

    const simulation = d3.forceSimulation(nodes)
      .force("link", d3.forceLink(links).id(d => d.id).distance(100))
      .force("charge", d3.forceManyBody().strength(-200))
      .force("center", d3.forceCenter(width / 2, height / 2));

    const link = svg.append("g")
      .selectAll("line")
      .data(links)
      .enter().append("line")
      .attr("stroke", "#94a3b8")
      .attr("stroke-width", 2);

    const node = svg.append("g")
      .selectAll("circle")
      .data(nodes)
      .enter().append("circle")
      .attr("r", 12)
      .attr("fill", "#4f46e5")
      .style("cursor", "pointer")
      .on("click", (event, d) => {
        model.set("selected_tag", { id: d.id, text: d.text || "" });
        model.save_changes();
      });

    const label = svg.append("g")
      .selectAll("text")
      .data(nodes)
      .enter().append("text")
      .text(d => d.id)
      .attr("font-size", "12px")
      .attr("dx", 15)
      .attr("dy", 4);

    simulation.on("tick", () => {
      link
        .attr("x1", d => d.source.x)
        .attr("y1", d => d.source.y)
        .attr("x2", d => d.target.x)
        .attr("y2", d => d.target.y);

      node
        .attr("cx", d => d.x)
        .attr("cy", d => d.y);

      label
        .attr("x", d => d.x)
        .attr("y", d => d.y);
    });
  }

  model.on("change:graph_data", draw);
  draw();
}

export default { render };
EOF

cat << 'EOF' > src/chunk_widget.py
import pathlib
import anywidget
import traitlets

_FRONTEND_DIR = pathlib.Path(__file__).parent / "frontend"

class ChunkGraphWidget(anywidget.AnyWidget):
    _esm = _FRONTEND_DIR / "chunk_graph.js"

    graph_data = traitlets.Dict({"nodes": [], "links": []}).tag(sync=True)
    selected_tag = traitlets.Dict({}).tag(sync=True)
EOF

cat << 'EOF' > src/graph_builder.py
import networkx as nx

class KnowledgeGraphBuilder:
    def __init__(self):
        self.graph = nx.Graph()

    def add_chunk_node(self, chunk_id: str, text: str, tags: list = None):
        self.graph.add_node(chunk_id, text=text, tags=tags or [])

    def add_relation(self, source_id: str, target_id: str, weight: float = 1.0):
        self.graph.add_edge(source_id, target_id, weight=weight)

    def to_json_data(self) -> dict:
        data = nx.node_link_data(self.graph)
        if "edges" in data and "links" not in data:
            data["links"] = data.pop("edges")
        return data
EOF

In [3]:
###TEST ALL ARCHITECTURE
import sys
import time
import subprocess
from pathlib import Path

# Assicura l'importazione dei moduli dalla cartella src/
if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

import qdrant_client
from qdrant_client.models import VectorParams, Distance, PointStruct
from langchain_ollama import OllamaEmbeddings, ChatOllama
from src.graph_builder import KnowledgeGraphBuilder
from src.chunk_widget import ChunkGraphWidget

print("=== 1. VERIFICA E AVVIO SERVIZI BACKEND ===")
# Controllo/Avvio di Qdrant e Ollama
def check_or_start_service(process_name, command):
    check = subprocess.run(["pgrep", "-f", process_name], capture_output=True)
    if not check.stdout:
        print(f"🚀 Avvio di {process_name} in corso...")
        subprocess.Popen(command, shell=True)
        time.sleep(2)
    else:
        print(f"✅ {process_name} è già attivo.")

check_or_start_service("qdrant", "cd ~/tesi_graphrag/data && nohup ~/.local/bin/qdrant > ~/tesi_graphrag/logs/qdrant.log 2>&1 &")
check_or_start_service("ollama serve", "nohup ~/.local/bin/ollama serve > ~/tesi_graphrag/logs/ollama.log 2>&1 &")

# Client Qdrant e modelli Ollama
qdrant_url = "http://localhost:6333"
client = qdrant_client.QdrantClient(url=qdrant_url)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
llm = ChatOllama(model="llama3.1", temperature=0)

print("\n=== 2. VECTOR STORE: INDICIZZAZIONE E SEARCH ===")
collection_name = "test_rag_chunks"
vector_dim = 768  # Dimensione per nomic-embed-text

# Reset/Creazione collezione Qdrant
if client.collection_exists(collection_name):
    client.delete_collection(collection_name)

client.create_collection(
    collection_name=collection_name,
    vectors_config=VectorParams(size=vector_dim, distance=Distance.COSINE)
)

# Chunk di test
test_chunks = [
    {"id": 0, "text": "GraphRAG combina l'uso di Vector DB e Grafi di Conoscenza per migliorare il recupero delle informazioni."},
    {"id": 1, "text": "Qdrant è un database vettoriale ad alte prestazioni ottimizzato per ricerche di similarità."},
    {"id": 2, "text": "NetworkX consente di rappresentare la struttura relazionale dei chunk sotto forma di grafo in Python."}
]

# Inserimento punti in Qdrant
points = []
for item in test_chunks:
    vector = embeddings.embed_query(item["text"])
    points.append(PointStruct(id=item["id"], vector=vector, payload=item))

client.upsert(collection_name=collection_name, points=points)
print(f"✅ Inseriti {len(points)} chunk vettorizzati su Qdrant.")

# Query semantica
query = "Come si collegano i grafi ai database vettoriali?"
query_vector = embeddings.embed_query(query)
search_results = client.query_points(
    collection_name=collection_name,
    query=query_vector,
    limit=2
).points

retrieved_texts = [hit.payload["text"] for hit in search_results]
print(f"🔍 Query: '{query}'")
print(f"📖 Chunk recuperati da Qdrant: {len(retrieved_texts)}")

print("\n=== 3. GENERAZIONE RISPOSTA LLM (RAG) ===")
prompt = f"Rispondi brevemente alla seguente domanda basandoti solo sul contesto fornito:\nDomanda: {query}\nContesto:\n" + "\n".join(retrieved_texts)
response = llm.invoke(prompt)
print(f"🤖 Risposta Llama 3.1:\n{response.content.strip()}")

print("\n=== 4. KNOWLEDGE GRAPH BUILDER & ANYWIDGET ===")
# Inizializzazione della classe backend per il grafo
builder = KnowledgeGraphBuilder()

# Inseriamo i nodi ed un arco di similarità basato sul recupero
for item in test_chunks:
    builder.add_chunk_node(chunk_id=f"chunk_{item['id']}", text=item["text"])

# Collegamento tra il chunk 0 e gli altri due
builder.add_relation("chunk_0", "chunk_1", weight=0.85)
builder.add_relation("chunk_0", "chunk_2", weight=0.78)

# Conversione e instanziazione Widget[cite: 1]
widget = ChunkGraphWidget()
widget.graph_data = builder.to_json_data()

print("✅ Test completato con successo. Visualizzazione del widget D3.js qui sotto:")
widget

=== 1. VERIFICA E AVVIO SERVIZI BACKEND ===
✅ qdrant è già attivo.
✅ ollama serve è già attivo.

=== 2. VECTOR STORE: INDICIZZAZIONE E SEARCH ===
✅ Inseriti 3 chunk vettorizzati su Qdrant.
🔍 Query: 'Come si collegano i grafi ai database vettoriali?'
📖 Chunk recuperati da Qdrant: 2

=== 3. GENERAZIONE RISPOSTA LLM (RAG) ===



KeyboardInterrupt



In [4]:
%%bash
cat << 'EOF' > ~/tesi_graphrag/src/config.py
import os

# ======================================================================
# SELEZIONE MODELLO LLM (Scommenta solo UNA delle opzioni)
# ======================================================================

LLM_MODEL = "llama3.2"       # OPZIONE A: Llama 3.2 (3B) -> Sviluppo fluido e bilanciato su CPU (Default)
# LLM_MODEL = "llama3.2:1b"  # OPZIONE B: Llama 3.2 (1B) -> Ultra-veloce per coding e debug rapido
# LLM_MODEL = "llama3.1"     # OPZIONE C: Llama 3.1 (8B) -> Massima qualità per i test finali


# ======================================================================
# SELEZIONE MODELLO EMBEDDING (Scommenta solo UNA delle opzioni)
# ======================================================================

EMBEDDING_MODEL = "nomic-embed-text"   # OPZIONE A: Nomic Embed Text (768 dim) -> Standard RAG
# EMBEDDING_MODEL = "mxbai-embed-large" # OPZIONE B: Mxbai Embed Large (1024 dim)
# EMBEDDING_MODEL = "all-minilm"        # OPZIONE C: All-MiniLM-L6-v2 (384 dim)


# ======================================================================
# CONFIGURAZIONE SERVIZI BACKEND E PARAMETRI CPU
# ======================================================================

QDRANT_URL = "http://localhost:6333"
OLLAMA_URL = "http://localhost:11434"
DEFAULT_KEEP_ALIVE = "2h"
DEFAULT_NUM_THREAD = 4
EOF

In [ ]:
import sys
from pathlib import Path

if str(Path.cwd()) not in sys.path:
    sys.path.append(str(Path.cwd()))

from langchain_ollama import ChatOllama
from langchain_community.embeddings import FastEmbedEmbeddings

# Import dei parametri centralizzati
from src.config import (
    LLM_MODEL, 
    EMBEDDING_MODEL, 
    DEFAULT_KEEP_ALIVE, 
    DEFAULT_NUM_THREAD
)

# 1. Istanzia l'LLM via Ollama
llm = ChatOllama(
    model=LLM_MODEL,
    keep_alive=DEFAULT_KEEP_ALIVE,
    num_thread=DEFAULT_NUM_THREAD,
    temperature=0
)

# 2. Istanzia l'Embedding locale via FastEmbed
embeddings = FastEmbedEmbeddings(model_name=EMBEDDING_MODEL)

print(f"✅ Setup completato | LLM: '{LLM_MODEL}' (Ollama) | Embeddings: '{EMBEDDING_MODEL}' (FastEmbed)")